In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('.\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [3]:
tabla_novedades = pd.read_sql_table('mensajeria_novedadesservicio',mensajeria)

tabla_novedades['tipo_novedad_id'].value_counts()
tabla_novedades.head(10)

,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7
5,9,2023-11-30 05:00:00+00:00,2,V,51,True,7
6,10,2023-11-30 05:00:00+00:00,2,V,51,True,7
7,11,2023-11-30 05:00:00+00:00,2,C,51,True,7
8,12,2023-11-30 05:00:00+00:00,1,B,29,True,7
9,15,2023-12-07 05:00:00+00:00,2,Me pinché,29,True,7


In [4]:
tabla_tiponovedad = pd.read_sql_table('mensajeria_tiponovedad', mensajeria)
tabla_tiponovedad.head(10)

,id,nombre
0,2,No puedo continuar
1,1,Novedades del servicio


In [5]:
dim_novedades= pd.merge(tabla_novedades,tabla_tiponovedad, left_on='tipo_novedad_id', right_on='id', how='left')
dim_novedades = dim_novedades.drop(columns=['id_y','mensajero_id','tipo_novedad_id'])
dim_novedades['key_novedad'] = range(1,len(dim_novedades)+1)
dim_novedades = dim_novedades.rename(columns={'id_x': 'novedad_id'})
dim_novedades.head(10)
#dim_novedades['descripcion'].value_counts()

,novedad_id,fecha_novedad,descripcion,servicio_id,es_prueba,nombre,key_novedad
0,4,2023-11-30 05:00:00+00:00,A,51,True,Novedades del servicio,1
1,5,2023-11-30 05:00:00+00:00,Halo,51,True,Novedades del servicio,2
2,6,2023-11-30 05:00:00+00:00,A,51,True,Novedades del servicio,3
3,7,2023-11-30 05:00:00+00:00,B,51,True,Novedades del servicio,4
4,8,2023-11-30 05:00:00+00:00,A,51,True,Novedades del servicio,5
5,9,2023-11-30 05:00:00+00:00,V,51,True,No puedo continuar,6
6,10,2023-11-30 05:00:00+00:00,V,51,True,No puedo continuar,7
7,11,2023-11-30 05:00:00+00:00,C,51,True,No puedo continuar,8
8,12,2023-11-30 05:00:00+00:00,B,29,True,Novedades del servicio,9
9,15,2023-12-07 05:00:00+00:00,Me pinché,29,True,No puedo continuar,10


In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
df = pd.read_sql_table('mensajeria_estadosservicio', mensajeria)
df[df['servicio_id'] == 21953].head(100).T


,213,530,98768,98771,98813,98920,98942,99192
id,98914,99019,98904,98907,98956,99063,99105,99343
fecha,2024-07-17 00:00:00,2024-07-17 00:00:00,2024-07-17 00:00:00,2024-07-17 00:00:00,2024-07-17 00:00:00,2024-07-17 00:00:00,2024-07-17 00:00:00,2024-07-17 00:00:00
hora,08:27:30,09:23:44,08:14:16,08:17:08,08:58:49,09:58:39,10:26:56,12:48:51
foto,foto,foto,foto,foto,foto,foto,foto,foto
observaciones,Recojo en Ppl,Recojo,Nuevo servicio solicitado,Con mensajero Asignado,Recojo b3,Esperando a q me autoricen la entrada,Rodolfo,Escritorio francy Yulieth
estado_id,3,4,1,2,3,3,5,6
servicio_id,21953,21953,21953,21953,21953,21953,21953,21953
es_prueba,True,False,True,True,True,True,False,False
foto_binary,None,None,None,None,None,None,None,None


In [7]:
dim_novedades['descripcion'].value_counts()

descripcion
Esperando en bodega                                                     39
En la espera de que alisten encomienda para enviar por terminal         38
Esperando despacho                                                      35
Lo toma otro compañero                                                  29
Esperando alistamiento                                                  19
                                                                        ..
Facturaron el refrigerante equivocado, se hará el cambio de producto     1
Edte drrvicio lo hace angelo                                             1
Edte lo hace csrlos                                                      1
Este lohace csrlos                                                       1
Este servicio rs moto csrguero                                           1
Name: count, Length: 3909, dtype: int64

In [8]:
dim_novedades.to_sql('dim_novedades', etl_conn, if_exists='replace', index=False)

208